In [1]:
import pandas as pd 
from IPython.display import display


# dudupped_data = pd.read_csv('data/pre_dup_train_data.tsv', sep='\t')
# dedupped_test  = pd.read_csv('data/pre_dup_test_data.tsv', sep='\t')
# print('Dedupe train data shape:', dudupped_data.shape)
# print('Dedupe test data shape:', dedupped_test.shape)

In [2]:
# Load data
train_df = pd.read_csv('data/de_dupped_train.tsv', sep='\t')
test_df = pd.read_csv('data/de_dupped_test.tsv', sep='\t')
print('Dedupe train data shape:', train_df.shape)
print('Dedupe test data shape:', test_df.shape)

/tmp/ipykernel_3333479/2433910744.py:2: DtypeWarning: Columns (0,1,2,5,6,7,14,15,16,17,18,20) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv('data/de_dupped_train.tsv', sep='\t')


Dedupe train data shape: (2115026, 24)
Dedupe test data shape: (6915, 24)


In [3]:
train_df['poet_era'].value_counts()

poet_era
العصر الحديث        808571
العصر العباسي       292258
العصر المملوكي      184438
العصر العثماني      168340
العصر الفاطمي       127222
العصر الأيوبي       120638
المغرب والأندلس     109076
العصر الأموي         74002
المخضرمون            39340
العصر الجاهلي        29406
العصر الأندلسي       20751
عصر بين الدولتين     20348
العصر الإسلامي        7450
عصرين                 1253
غير محدد               493
Name: count, dtype: int64

In [49]:


# 1. Merge `gnre` into `genre` if genre is missing or empty
for df in [train_df, test_df]:
    df['genre'] = df['genre'].fillna('').astype(str).str.strip()
    df['gnre'] = df['gnre'].fillna('').astype(str).str.strip()
    df['genre'] = df.apply(lambda row: row['gnre'] if not row['genre'] else row['genre'], axis=1)

# 2. Strip leading/trailing spaces from all string columns
def strip_all_columns(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.strip()
    return df

train_df = strip_all_columns(train_df)
test_df = strip_all_columns(test_df)

# 3. Define mappings
country_names = [
    'مصر', 'لبنان', 'المغرب', 'سوريا', 'تونس', 'فلسطين', 'موريتانيا', 'العراق',
    'اليمن', 'ليبيا', 'السعودية', 'السودان', 'الجزائر', 'الأردن', 'عمان',
    'الإمارات', 'طباعة', 'أجمل الابيات'
]

era_mapping = {
    'الحديث': 'العصر الحديث',
    'العثماني': 'العصر العثماني',
    'المملوكي': 'العصر المملوكي',
    'العباسي': 'العصر العباسي',
    'الأموي': 'العصر الأموي',
    'الفاطمي': 'العصر الفاطمي',
    'الأيوبي': 'العصر الأيوبي',
    'قبل الإسلام': 'العصر الجاهلي',
    'العصر الايوبي': 'العصر الأيوبي',
    'العصر الاموي': 'العصر الأموي',
    'الدولة الايوبية': 'العصر الأيوبي',
    'الدولة الفاطمية': 'العصر الفاطمي',
    'الدولة المملوكية': 'العصر المملوكي',
    'بين الدولتين' : 'عصر بين الدولتين',
    'الإسلامي': 'العصر الإسلامي',
    'العصر الاسلامي': 'العصر الإسلامي',
    'قبل الإسلام': 'العصر الجاهلي',
    
    # Standardize all to المخضرمون
    'المخضرمون': 'المخضرمون',
    'المخضرمين': 'المخضرمون',
    'الشعراء المخضرمون': 'المخضرمون',
}

# 4. Normalize poet_era and fix location
def normalize_poet_era_and_location(df):
    for idx, row in df.iterrows():
        era = row['poet_era']
        location = row['location']
        
        # If poet_era is a country name
        if era in country_names:
            # Update location if it's empty or different
            if not location or location.strip() == '':
                df.at[idx, 'location'] = era
            elif era not in location:
                df.at[idx, 'location'] = era  # overwrite to keep clean

            # Set poet_era to غير محدد
            df.at[idx, 'poet_era'] = 'غير محدد'
        elif era in era_mapping:
            df.at[idx, 'poet_era'] = era_mapping[era]
    return df

train_df = normalize_poet_era_and_location(train_df)
test_df = normalize_poet_era_and_location(test_df)

# 5. Drop gnre column
train_df.drop(columns=['gnre'], inplace=True)
test_df.drop(columns=['gnre'], inplace=True)



/tmp/ipykernel_2047242/1903982380.py:4: DtypeWarning: Columns (0,1,2,3,5,6,7,9,13,14,15,16,17,19) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv('data/de_dupped_train.tsv', sep='\t')


✅ Cleaned data saved to data/cleaned_train.tsv and data/cleaned_test.tsv


In [27]:
import os
from IPython.display import display

# Initialize an empty dictionary to store dataframes
final_data = {}

# Directory containing the TSV files
directory_path = "/path/to/jais_efforts/poetry/data/processed"

# Iterate through each file in the directory
for file_name in os.listdir(directory_path):
    for sub_dir in os.listdir(directory_path):
        sub_dir_path = os.path.join(directory_path, sub_dir)
        if os.path.isdir(sub_dir_path):  # Check if it's a directory
            test_file = os.path.join(sub_dir_path, "test.tsv")
            train_file = os.path.join(sub_dir_path, "train.tsv")
            
            # Check if both files exist
            if os.path.exists(test_file) and os.path.exists(train_file):
                # Load and display the test file
                print(f"Displaying test file from {sub_dir}:")
                test_df = pd.read_csv(test_file, sep='\t')
                display(test_df.head())
                
                # Load and display the train file
                print(f"Displaying train file from {sub_dir}:")
                train_df = pd.read_csv(train_file, sep='\t')
                display(train_df.head())

Displaying test file from poem_text__poet_era:


,poem_text,poet_era,poem_text_num_tokens,poet_era_num_tokens
0,1\n\nحَيّـاكُمُ اللَـهُ أَحيـوا العِلمَ وَالأَ...,العصر الحديث,3625,3
1,1\n\nغـابَ الأَديـبُ أَديـبُ مِصرٍ وَاِختَفى\n...,العصر الحديث,229,3
2,1\n\nعُثمـانُ إِنَّكَ قَد أَتَيتَ مُوَفَّقاً\n...,العصر الحديث,188,3
3,1\n\nإِنَّ عَضــّيكَ يــا أَخــي بِــالمَلامِ\...,العصر الحديث,809,3
4,1\n\nمِن واجِدٍ مُنَقِّرِ المَنامِ\n\nطَريدَ د...,العصر الحديث,762,3


Displaying train file from poem_text__poet_era:


,poem_text,poet_era,poem_text_num_tokens,poet_era_num_tokens
0,سَلامٌ في الصَحيفَةِ مِن لَقيطٍ\nإِلى مَن بِال...,العصر الجاهلي,212,3
1,يا دارَ عَمرَةَ مِن مُحتَلِّها الجَرَعا\nهاجَت...,العصر الجاهلي,3905,3
2,وَخانَنا خَوّانٌ في اِرتِباعِنا\nفَاِنفَدَّ لِ...,العصر الجاهلي,56,3
3,تَنَحَّ إِلَيكُم يا اِبنَ كوزٍ فَإِنَّنا\nوَإِ...,العصر الجاهلي,65,3
4,تُطارِحُهُ الأَنسابُ حَتّى رَدَدنَهُ\nإِلى نَس...,العصر الجاهلي,65,3


Displaying test file from poem_text__overall_explanation:


...

,poet_name,poet_era,poet_name_num_tokens,poet_era_num_tokens
0,لقيط بن يعمر الإيادي,العصر الجاهلي,7,3
1,لقيط بن يعمر الإيادي,العصر الجاهلي,7,3
2,لقيط بن يعمر الإيادي,العصر الجاهلي,7,3
3,زبان بن سيار الفزاري,العصر الجاهلي,8,3
4,زبان بن سيار الفزاري,العصر الجاهلي,8,3


Displaying test file from poem_text__meter:


,poem_text,meter,poem_text_num_tokens,meter_num_tokens
0,1\n\nحَيّـاكُمُ اللَـهُ أَحيـوا العِلمَ وَالأَ...,البسيط,3625,2
1,1\n\nغـابَ الأَديـبُ أَديـبُ مِصرٍ وَاِختَفى\n...,الكامل,229,2
2,1\n\nعُثمـانُ إِنَّكَ قَد أَتَيتَ مُوَفَّقاً\n...,الكامل,188,2
3,1\n\nإِنَّ عَضــّيكَ يــا أَخــي بِــالمَلامِ\...,الخفيف,809,2
4,1\n\nمِن واجِدٍ مُنَقِّرِ المَنامِ\n\nطَريدَ د...,الرجز,762,3


Displaying train file from poem_text__meter:


,poem_text,meter,poem_text_num_tokens,meter_num_tokens
0,فَأُبنا وَآبوا كُلَّنا بِمَضيضَةٍ\nمُهَمَّلَةٍ...,الطويل,60,2
1,يَدٌ مِن بَعيدٍ أَو قَريبٍ أَتَت بِهِ\nشَآمِيّ...,الطويل,61,2
2,إِذا ما رَآني الناسُ قالوا أَلَم تَكُن\nحَديثا...,الطويل,50,2
3,لا عَجيبٌ فيما رَأَيتِ وَلَكِن\nعَجَبٌ مِن تَف...,الخفيف,50,2
4,وَالفَريدَ المُسَفَّعَ الوَجهِ ذا الجُدَّةِ\nي...,الخفيف,51,2


In [4]:
import yaml
import re
from pathlib import Path

def remove_numbering_from_texts(data):
    for lang, templates in data.items():
        for template in templates:
            if isinstance(template.get("text"), list):
                cleaned_texts = [
                    re.sub(r"^\s*\d+\.\s*", "", text)
                    for text in template["text"]
                ]
                template["text"] = cleaned_texts
    return data

def main(input_path: str, output_path: str = None):
    input_path = Path(input_path)
    output_path = Path(output_path) if output_path else input_path

    # Load YAML
    with input_path.open("r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    # Clean numbering
    cleaned_data = remove_numbering_from_texts(data)

    # Save YAML
    with output_path.open("w", encoding="utf-8") as f:
        yaml.dump(cleaned_data, f, allow_unicode=True, sort_keys=False)


main('/path/to/templates/poetry_analysis_all.yaml', 
     '/path/to/templates/poetry_analysis_all_final.yaml')